## Exercise 1: Calculate pressure and fluid composition from melt and fluid inclusion data

In this exercise we will calculate pressures and fluid compositions from MI and FI data from the 2018 Lower East Rift Zone eruption of Kı̄lauea (Wieser et al., 2021; DeVitre & Wieser, 2024) using the tools VESIcal (Iacovino et al., 2021), DiadFit (Wieser & DeVitre, 2024), VolFe (Hughes et al., 2025), and Thermobar (Wieser et al., 2022) through the VICTOR platform (Lev et al., 2025). 

For melt compositions, this is based on the fact that - for a particular major/minor element composition, temperature, and volatile content (partcuarly H<sub>2</sub>O and CO<sub>2</sub>) of the melt - there is a unique pressure at which the melt will be vapor saturated, called the pressure of vapor saturation, and the coexisting fluid will have a unique composition (see Hughes et al., 2024 for more details). This is often applied to MI data to calculate pressures of magma storage or submarine matrix glass (MG) data to estimate eruption pressures, which in both cases can be converted to depths given overburden density. The fluid composition is sometimes used to estimate the composition of MI-hosted bubbles to reconstruct MI compositions at entrapment. 

Alternatively, for fluid compositions, the density of CO<sub>2</sub> in a fluid co-existing with a melt is highly pressure-sensitive. Hence, when combined with an entrapment temperature, the CO<sub>2</sub> density can be converted to pressure using an equation of state for CO<sub>2</sub> (see ??? for more details). This is often applied to FI data to also calculate pressures of magma storage.

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b>🐣 &nbsp; To run locally</b><br> See the notebook <b><a href="0_Getting_Started.ipynb">0. Getting Started</a></b>. Always remember to ensure this notebook is running in the volatiles-gs-workshop26 conda environment!

<em>If you are running this on the VICTOR platform, all packages are already installed and you can skip this step. More information on VICTOR can be found at <a href="https://docs.victorproject.orgr/en/latest/index.html">https://docs.victorproject.orgr/en/latest/index.html</em></a>
</div>

## 1. Introduction
### 1.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import matplotlib.pyplot as plt
import VolFe as vf
import VESIcal as vc
import Thermobar as tb
import evo

import os
os.makedirs("output", exist_ok=True)

When reporting calculations in manuscripts, it's important to know which version of the Python package the results you are showing used - so we can output those below. This notebook was created using VolFe: 1.0.2 , VESIcal: 1.2.12 , Thermobar: 1.0.73 , pandas: 2.2.3, EVo: 1.1.0 (at git sha 2487939e18d98292f9b8f3a1f0dea04b707bd89f), and Sulfur_X 1.2 (git sha d605a34135c906c00f26de77b7e7f9abdb05d86f).

In [ ]:
# print("VolFe " + vf.__version__ +", " +
#       "VESIcal ", VESIcal '+vc.__version__, ', Thermobar '+tb.__version__ '')
      
print(
    f"VolFe: {vf.__version__}",
    f"\nVESIcal: {vc.__version__}",
    f"\nThermobar: {tb.__version__}",
    f"\npandas: {pd.__version__}",
    )

### 1.2 Import data

Typically, you'll have a spreadsheet full of either melt or fluid inclusion analyses to process. In this case, we've created simplified csv files derived from the spreadsheets provided in Supplementary Materials of the original papers (i.e., some columns and rows have been deleted to make them easier to use).

<b>Melt Inclusions</b>
<a href="files/wieser2021_data_simple.csv">files/wieser2021_data_simple.csv</a> is derived from <i>Supplement_Wieser_et_al_G3.xlsx</i> from Wieser et al. (2021) combining data in the 'Melt_Inclusions' and 'Matrix_Glasses' sheets. Only the PEC-corrected data is included, as well as the bubble+glass value for CO<sub>2</sub>. The S data is unpublished data provided by Wieser (pers. comm.).

<b>Fluid Inclusions</b>
<i>??? something about FI data ???</i>

In [ ]:
# import melt inclusion dataset
MI = pd.read_csv("data/wieser2021_data_simple.csv")

# import fluid inclusion dataset

We can plot the data to see how it looks - like the H<sub>2</sub>O and CO<sub>2</sub> content of melt inclusions and matrix glass.

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4,4))

df = MI
# subset by type
df_MI = df[df['type'] == 'MI']
df_MG = df[df['type'] == 'MG']

# MI: open circles
ax1.plot(df_MI['H2O (wt%) (wt%, PEC-corr) MI'], df_MI['Total CO2 (PEC-corrected) ppm'], 'ok', mfc='white', label = "MI")
# MG: x symbols
ax1.plot(df_MG['H2O (wt%) (wt%, PEC-corr) MI'], df_MG['Total CO2 (PEC-corrected) ppm'], 'xk', label="MG")

# plot styling
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax1.set_ylabel('CO$_2$ [m] (ppm)')
ax1.legend(frameon=False)

H<sub>2</sub>O contents of melt inclusions commonly show signs of "resetting", wherein dissolved H<sub>2</sub>O concentrations are <em>lower</em> than those at the time of entrapment due to diffusive requilibration throught the crystal host during crustal magma storage (Wieser et al. 2021). For this exercise, we set the MI H2O contents to 0.5 wt% as in Wieser et al. (2021).

In [ ]:
for row in range(0,len(MI),1):
    if MI.loc[row,'type'] == 'MI':
        MI.loc[row,'H2O (wt%) (wt%, PEC-corr) MI'] = 0.5

## 2. Calculate temperature using Thermobar

A temperature is requied to calculate pressures and fluid compositions from MI/MG data. Thermobar has a huge variety of thermometers available to choose from. 

More information on Thermobar can be found at https://thermobar.readthedocs.io/en/latest/

Run the next cell to see the options available in Thermobar just for liquid-only thermometers.

In [ ]:
help(tb.calculate_liq_only_temp)

Following Wieser et al. (2021), we calculate the temperature using the MgO-liquid thermometer of Helz & Thornber (1987) implemented in Thermobar. 

Thermobar requires certain column names to recognise what each column contains - so first we'll need to rename the columns to be compatible with Thermobar. Thermobar also assumes the composition is in wt%, so anything that isn't would need to be converted. Then we can run the calculation and add the results to the original dataframe!

In [ ]:
# renames column names to be compatible with Thermobar
MI_tb = MI.rename(columns = {"SiO2 (wt%, PEC-corr) MI":"SiO2_Liq","TiO2 (wt%, PEC-corr) MI":"TiO2_Liq","Al2O3 (wt%, PEC-corr) MI":"Al2O3_Liq","FeO (wt%, PEC-corr) MI":"FeOt_Liq","MnO (wt%, PEC-corr) MI":"MnO_Liq","MgO (wt%, PEC-corr) MI":"MgO_Liq","CaO (wt%, PEC-corr) MI":"CaO_Liq",
                                               "Na2O (wt%, PEC-corr) MI":"Na2O_Liq","K2O (wt%, PEC-corr) MI":"K2O_Liq","P2O5 (wt%, PEC-corr) MI":"P2O5_Liq","H2O (wt%) (wt%, PEC-corr) MI":"H2O_Liq"})

# calculate temperature in celcius (hence -273.15) using MgO-Lq thermometer of Helz & Thornber (1987) using Thermobar
T_C = tb.calculate_liq_only_temp(liq_comps=MI_tb, equationT="T_Helz1987_MgO")-273.15

# adds temperatures to original dataframe
if "T_C" not in MI.columns:
    MI.insert(1,"T_C",T_C)

# save your results
MI.to_csv("output/wieser2021_w_temperatures.csv")

We can plot the MgO content against the temperature to see the correlation.

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4,4))

df = MI
# subset by type
df_MI = df[df['type'] == 'MI']
df_MG = df[df['type'] == 'MG']

ax1.plot(df_MI['MgO (wt%, PEC-corr) MI'], df_MI['T_C'], 'ok', mfc='blue', label="MI")
ax1.plot(df_MG['MgO (wt%, PEC-corr) MI'], df_MG['T_C'], 'xk', color='blue',label="MG")

ax1.set_xlabel('MgO[m] (wt%)')
ax1.set_ylabel('T (°C)')
ax1.legend(frameon=False)

## 3. Calculate saturation pressures and equilibrium fluid compositions

### 3.1 VESIcal

We can now calculate the pressure of vapor saturation and fluid composition using one of several H<sub>2</sub>O, CO<sub>2</sub>, or mixed H<sub>2</sub>O-CO<sub>2</sub> solubility models in VESIcal. All VESIcal models assume the fluid and melt only contain oxidised C-O-H-bearing species CO<sub>2</sub> and H<sub>2</sub>O. 

More information on VESIcal can be found at https://vesical.readthedocs.io/en/latest/

There are multiple model options available in VESIcal for the H<sub>2</sub>O and CO<sub>2</sub> solubility. Run the cell below to list the models available. In any VESIcal calculation call, use the argument ``model="your-model-of-interest"``, which we will demonstrate below.

In [ ]:
vc.get_model_names()

Here we'll use the model of Iacono-Marziano et al. (2021) ``"IaconoMarziano"``. <i>??? Can use MagmaSat if it'll be running on VICTOR ??? </i>

Note that how the melt composition is normalised is important for the results and different H<sub>2</sub>O-CO<sub>2</sub> models use different normalisation routines (further explanations can be found in Iacovino et al., 2021, and Wieser et al., 2021). IaconoMarziano uses the 'additionalvolatiles' normalisation routine, whereby the anhydrous melt composition is normalised to 100 wt%; CO<sub>2</sub> and H<sub>2</sub>O are added on top (i.e., the sum is greater than 100 wt%); and then everything is renormalised to 100 wt%.

Now we can run VESIcal using our chosen H<sub>2</sub>O-CO<sub>2</sub> model. As with Thermobar, VESIcal requires certain (but different) column headers to understand the input data. This is why we changed the column headers in the notebook rather than in the spreadsheet. Also, CO<sub>2</sub> needs to be in wt% not ppm, so we convert it. Then we use the <i>calculate_equilibrium_fluid_comp</i> function to calculate both pressure and fluid composition.

In [ ]:

# renames column names to be compatible with VESIcal
oxides = ["SiO2", "TiO2", "Al2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5"]

rename_dict = {f"{col} (wt%, PEC-corr) MI": col for col in oxides}
rename_dict["H2O (wt%) (wt%, PEC-corr) MI"] = "H2O"
rename_dict["Sample Name"] = "Label"

MI_vc = MI.rename(columns=rename_dict)
MI_vc.index.name = "Label"

# converts CO2 from ppm to wt% and uses header compatible with VESIcal
MI_vc['CO2'] = MI_vc['Total CO2 (PEC-corrected) ppm']/10000.


MI_vc_bf = vc.BatchFile(filename=None, dataframe=MI_vc)
results_pvsat_vc = MI_vc_bf.calculate_equilibrium_fluid_comp(temperature='T_C', model="IaconoMarziano")

# save your results
results_pvsat_vc.to_csv("output/results_pvsat_vc.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_pvsat_vc

And we plot to see how the melt volatile content changes with the calculated pressure and what the fluid composition is doing as well!

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,4))

df = results_pvsat_vc
# subset by type
df_MI = df[df['type'] == 'MI']
df_MG = df[df['type'] == 'MG']

ax1.plot(df_MI['H2O'], df_MI['SaturationP_bars_VESIcal'], 'ok', mfc='red', label = "MI")
ax1.plot(df_MG['H2O'], df_MG['SaturationP_bars_VESIcal'], 'xk', color='red', label = "MG")
ax2.plot(df_MI['CO2']*10000, df_MI['SaturationP_bars_VESIcal'], 'ok', mfc='red')
ax2.plot(df_MG['CO2']*10000, df_MG['SaturationP_bars_VESIcal'], 'xk', color='red')
ax3.plot(df_MI['XCO2_fl_VESIcal'], df_MI['SaturationP_bars_VESIcal'], 'dk', mfc='red')
ax3.plot(df_MG['XCO2_fl_VESIcal'], df_MG['SaturationP_bars_VESIcal'], 'xk', color='red')

ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O[m] (wt%)')
ax2.set_xlabel('CO$_2$[m] (ppm)')
ax3.set_xlabel('CO$_2$[f] (mol. frac.)')
ax1.legend(frameon=False)

plt.tight_layout()

### 3.3 VolFe

We can also calculate the pressure of vapor saturation and fluid composition using VolFe, which assumes the fluid and melt contain a variety of reduced and oxidised C-O-H-S-bearing species. This needs additional information, such as some estimate of an oxygen fugacity related variable (e.g., fO2, deltaFMQ, deltaNNO, Fe3+/FeT, or S6+/ST). We'll use a value of deltaFMQ+0.2 for all MI and MG, which is within the range described in Wieser et al. (2021), but in other examples this can be specified for each composition individually.

More information on VolFe can be found at https://volfe.readthedocs.io/en/latest/

There are lots of different options available in VolFe. Similar to VESIcal, there are different H2O and CO2 solubility models, but there are also different sulfide and sulfate solubility models and relationships between Fe3+/FeT and fO2 to choose from. Run the cell below to see all the parameters that can be changed in VolFe.

In [ ]:
help(vf.make_df_and_add_model_defaults)

Here we'll focus on the H2O, CO2, sulfide, and sulfate solubility options:

In [ ]:
# H2O solubility model
help(vf.C_H2O)

In [ ]:
# CO2 solubility model
help(vf.C_CO3)

In [ ]:
# sulfide solubility model
help(vf.C_S)

In [ ]:
# sulfate solubility model
help(vf.C_SO4)

We'll choose to use water solubility model from Fig. S2 of Hughes et al. (2024); CO2 solubility model from eq. (7) from Dixon (1997); sulfide solubility model from eq. (10.34) of O'Neill (2021); and sulfate solubility model from eq. (12a) of O'Neill & Mavrogenes (2022) - everything else will use the default options in VolFe. All options used are outputted in the results.

In [ ]:
# choose the options I want - everything else will use the default options
models_vf = [['water','Basalt_Hughes24'],['carbon dioxide','Basalt_Dixon97'],['sulfide','ONeill21dil'],['sulfate','ONeill22dil']]# IS there any other models and parameters that the users can choose/input? And how?

# turn to dataframe with correct column headers and indexes    
models_vf = vf.make_df_and_add_model_defaults(models_vf)

VolFe also uses different column names...

In [ ]:
MI_vf = MI.rename(columns = {'Sample Name':'Sample',"SiO2 (wt%, PEC-corr) MI":"SiO2","TiO2 (wt%, PEC-corr) MI":"TiO2","Al2O3 (wt%, PEC-corr) MI":"Al2O3","FeO (wt%, PEC-corr) MI":"FeOT","MnO (wt%, PEC-corr) MI":"MnO","MgO (wt%, PEC-corr) MI":"MgO","CaO (wt%, PEC-corr) MI":"CaO",
                                               "Na2O (wt%, PEC-corr) MI":"Na2O","K2O (wt%, PEC-corr) MI":"K2O","P2O5 (wt%, PEC-corr) MI":"P2O5","H2O (wt%) (wt%, PEC-corr) MI":"H2O", "Total CO2 (PEC-corrected) ppm":'CO2ppm',"S_MI_PEC EPMA":"STppm"})

And add DFMQ to each row...

In [ ]:
# Adds oxygen fugacity constraint to each row in the dataframe
MI_vf['DFMQ'] = 0.20

And then we run the calculations...

In [ ]:
# runs the calculation
results_pvsat_vf = vf.calc_Pvsat(MI_vf, models= models_vf)

# save the results
results_pvsat_vf.to_csv("output/results_pvsat_vf.csv")

... and compare them to the VESIcal results:

In [ ]:
# Would it be possible to differentiate the VolFe resutls of MG and MI as well? I think it would be useful to illustrate your point about the mol fraction CO2>1

In [ ]:
fig, ((ax1, ax2, ax3),(ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(12,8))

df = results_pvsat_vc
# subset by type
df_MI = df[df['type'] == 'MI']
df_MG = df[df['type'] == 'MG']

ax1.plot(df['H2O'], df['SaturationP_bars_VESIcal'], 'ok', mfc='red', label ="VESIcal_IM")
ax2.plot(df['CO2']*10000, df['SaturationP_bars_VESIcal'], 'ok', mfc='red')
ax3.plot(df_MI['XCO2_fl_VESIcal'], df_MI['SaturationP_bars_VESIcal'], 'dk', mfc='red', label = "MG")
ax3.plot(df_MG['XCO2_fl_VESIcal'], df_MG['SaturationP_bars_VESIcal'], 'xk', color='red', label = "MI")

df = results_pvsat_vf

ax1.plot(df['H2OT-eq_wtpc'], df['P_bar'], 'ok', mfc='purple', label = "VolFe")
ax2.plot(df['CO2T-eq_ppmw'], df['P_bar'], 'ok', mfc='purple')
ax3.plot(df['xgCO2_mf'], df['P_bar'], 'dk', mfc='purple')

ax4.plot(df['ST_ppmw'], df['P_bar'], 'ok', mfc='purple')
ax5.plot(df['Fe3+/FeT'], df['P_bar'], 'ok', mfc='purple')
ax6.plot(df['xgSO2_mf']/df['xgCO2_mf'], df['P_bar'], 'dk', mfc='purple')

ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O[m] (wt%)')
ax2.set_xlabel('CO$_2$[m] (ppm)')
ax3.set_xlabel('CO$_2$[f] (mol. frac.)')
ax4.set_ylabel('P (bars)')
ax4.set_xlabel('S[m] (ppm)')
ax5.set_xlabel('Fe$^{3+}$/Fe$_T$[m] (mole frac.)')
ax6.set_xlabel('SO$_2$/CO$_2$[f] (mole frac.)')
ax1.legend(frameon=False)

plt.tight_layout()

What?! Negative pressures?! CO<sub>2</sub> vapor compositions more than 1?!

These are for MG with ~65 wt% SiO<sub>2</sub>, where the eq. (7) from Dixon (1997) parameterisation is not appropriate for and actually ends up predicting a negative solubility function... we'll remove those composition for further comparisons! But it's important to check where the calibrations are appropriate!

In [ ]:
results_pvsat_vf = results_pvsat_vf[results_pvsat_vf['SiO2_wtpc'] < 60.]

### OTHER MODELS?

### ??? 1.3 FLUID INCLUSION PART ???